In [1]:
import cvxpy as cp
import numpy as np
from scipy.optimize import minimize

In [2]:
# 1. Setup the market data (Covariance Matrix)
mu = np.array([0.3, 0.5, 0.2])
sigma = np.array([
    [0.10, 0.02, 0.04],
    [0.02, 0.08, 0.01],
    [0.04, 0.01, 0.12]
])
mu_p = 0.8

The mean-variance problem subject to a target expected return $\mu_p$ and budget constraint:
$$\min_w \frac{1}{2}w^T\Sigma w \quad \text{s.t.} \quad w^T\mu = \mu_p, \ \ \mathbf{1}^Tw = 1$$
The Lagrangian with two multipliers:
$$\mathcal{L} = \frac{1}{2}w^T\Sigma w - \lambda(w^T\mu - \mu_p) - \gamma(\mathbf{1}^Tw - 1)$$
First order condition: $\Sigma w - \lambda\mu - \gamma\mathbf{1} = 0 \implies w = \Sigma^{-1}(\lambda\mu + \gamma\mathbf{1})$

Solve for the multipliers by imposing the two constraints, using the three frontier scalars — $A = \mathbf{1}^T\Sigma^{-1}\mathbf{1}$, $B = \mathbf{1}^T\Sigma^{-1}\mu$, $C = \mu^T\Sigma^{-1}\mu$, and $D=AC-B^2$:

Return constraint $w^T\mu = \mu_p$​:
$$\mu^T\Sigma^{-1}(\lambda\mu + \gamma\mathbf{1}) =\mu_p, \qquad \lambda \mu^T\Sigma^{-1}\mu + \gamma\,\mu^T\Sigma^{-1}\mathbf{1} = \mu_p, \qquad \lambda C + \gamma B = \mu_p​$$
Budget constraint $\mathbf{1}^Tw = 1$:
$$\mathbf{1}^T\Sigma^{-1}(\lambda\mu + \gamma\mathbf{1}) = 1, \qquad \lambda\mathbf{1}^T\Sigma^{-1}\mu + \gamma\mathbf{1}^T\Sigma^{-1}\mathbf{1} =1,\qquad \lambda B + \gamma A = 1$$

Two linear equations in the two unknowns $\lambda$, $\gamma$. Stach them:
$$\begin{pmatrix} C & B \\ B & A \end{pmatrix}\begin{pmatrix} \lambda \\ \gamma \end{pmatrix} = \begin{pmatrix} \mu_p \\ 1 \end{pmatrix}$$
Solving it. Invert the $2\times2$:
$$\begin{pmatrix}\lambda\\\gamma\end{pmatrix} = \frac{1}{AC - B^2}\begin{pmatrix} A & -B \\ -B & C\end{pmatrix}\begin{pmatrix}\mu_p\\1\end{pmatrix} = \frac{1}{D}\begin{pmatrix} A\mu_p - B \\ C - B\mu_p\end{pmatrix}$$
The closed-form weights:
$$\boxed {\omega=\frac{C \Sigma^{-1} \mathbf{1} - B \Sigma^{-1} \mu}{D} + \mu_p \frac{A \Sigma^{-1} \mu - B \Sigma^{-1} \mathbf{1}}{D}}$$

In [3]:
def min_variance_target_return(mu, sigma, mu_p):
    """
    Closed-form minimum-variance portfolio for a target expected return,
    subject to budget and return constraints (no inequality constraints):

        min  (1/2) w^T Sigma w
        s.t. w^T mu = mu_p,  1^T w = 1

    Solution:
        w* = (1/D) [ (A*mu_p - B) Sigma^-1 mu  +  (C - B*mu_p) Sigma^-1 1 ]
    where A = 1^T Sinv 1, B = 1^T Sinv mu, C = mu^T Sinv mu, D = A*C - B^2.

    Allows short positions (long-short). Returns weights summing to 1.

    Parameters
    ----------
    mu    : (n,) expected returns
    sigma : (n, n) covariance matrix, assumed SPD
    mu_p  : scalar target expected return

    Returns
    -------
    w : (n,) optimal weights
    """
    mu = np.asarray(mu, dtype=float).ravel()
    sigma = np.asarray(sigma, dtype=float)
    n = mu.shape[0]
    ones = np.ones(n)

    # The one habit worth locking in: solve, don't invert. Notice I used np.linalg.solve(sigma, ones) rather than np.linalg.inv(sigma) @ ones.
    # They're mathematically identical, but solve is faster and numerically more stable — it uses an LU factorization under the hood instead of forming the full inverse, which squares the conditioning damage.
    # This is the exact Σ^{-1}-amplification point: never form Σ^{-1} if you only need it applied to a vector.
    x = np.linalg.solve(sigma, mu)      # Sigma^-1 mu
    y = np.linalg.solve(sigma, ones)    # Sigma^-1 1

    # Frontier scalars
    A = ones @ y                        # 1^T Sigma^-1 1
    B = ones @ x                        # 1^T Sigma^-1 mu   ( = mu^T Sigma^-1 1 )
    C = mu @ x                          # mu^T Sigma^-1 mu
    D = A * C - B * B                   # determinant of the 2x2 multiplier system

    if D <= 0 or not np.isfinite(D):
        raise ValueError(
            f"degenerate frontier (D={D:.3e}): mu and 1 are nearly collinear "
            "or Sigma is ill-conditioned"
        )

    lam = (A * mu_p - B) / D            # multiplier on the return constraint
    gam = (C - B * mu_p) / D            # multiplier on the budget constraint

    w = lam * x + gam * y              # w = Sigma^-1 (lam*mu + gam*1)
    return w
min_variance_target_return(mu, sigma, mu_p)

array([-0.16666667,  2.05555556, -0.88888889])

The moment you add no-shorting ($w \geq 0$) or a leverage cap, you reintroduce inequality constraints, complementary slackness kicks in, and you lose the closed form.

In [4]:
def min_variance_scipy(sigma, bounds=None):
    num_assets = len(sigma)
    # 2. Define the Objective Function 
    # The Python `@` operator handles the matrix multiplication: w^T * Sigma * w
    def objective_function(w, cov_matrix):
        return w.T @ cov_matrix @ w

    # 3. Define the Jacobian (The First-Order Derivative)
    # Passing the exact gradient prevents the solver from wasting time guessing it.
    # The Jacobian Advantage: If you do not provide the jac argument, scipy will use "finite differences" to guess the gradient by perturbing the weights slightly thousands of times. Because you mathematically know that the derivative is $2 \Omega w$, passing that function explicitly cuts the solver's computational time in half. This is pure execution alpha.
    def jacobian_derivative(w, cov_matrix):
        return 2 * cov_matrix @ w

    # 4. Define the Constraints
    # Budget Constraint: sum(w) - 1.0 = 0
    def budget_constraint(w):
        return np.sum(w) - 1.0

    # SciPy expects constraints as a tuple of dictionaries
    constraints = ({'type': 'eq', 'fun': budget_constraint})

    # 5. Define the Bounds
    # Long-Only Constraint: Each weight must be between 0.0 and 1.0
    bounds = tuple((0.0, 1.0) for _ in range(num_assets))

    # 6. Set the Initial Guess
    # The solver needs a starting point. Equal weighting is the standard default.
    initial_guess = np.ones(num_assets) / num_assets

    # 7. Run the Optimizer
    # The SLSQP Engine: You use SLSQP specifically because it is one of the only algorithms in SciPy that can handle both Bounds (e.g., w >= 0) and Constraints (e.g., sum(w) = 1) simultaneously for quadratic functions. trust-constr` is the only real alternative that does, but SLSQP is faster and more standard for smooth QP-shaped problems this size.
    result = minimize(
        fun=objective_function,
        x0=initial_guess,
        args=(sigma,),       # Passes the covariance matrix to our functions
        method='SLSQP',           # The algorithm used for constrained optimization
        jac=jacobian_derivative,  # Our custom gradient function
        bounds=bounds,
        constraints=constraints,
        options={'ftol': 1e-12, 'maxiter': 500}
    )

    # 8. Output the Result
    # Always check res.success and inspect res.x for sanity (weights summing to 1, no wild values). SLSQP can silently return a non-converged point.
    if result.success:
        print(f"Optimal Weights: {result.x.round(4)}")
        print(f"Minimum Variance Achieved: {result.fun:.4f}")
    else:
        print("Optimization Failed:", result.message)
min_variance_scipy(sigma)

Optimal Weights: [0.2632 0.4795 0.2573]
Minimum Variance Achieved: 0.0462


SLSQP is a general nonlinear solver, not a QP solver. It doesn't know your problem is convex, so it can be slower and less robust than a true QP method on large/ill-conditioned problems.

For a learning toolkit at small-to-medium n it's perfectly fine, and using it teaches you the constraint-specification mechanics. But have a solver='cvxpy' path in mind for when you scale up or hit a near-singular Σ; CVXPY+OSQP will recognize the QP structure and be far more stable.

In [5]:
def min_variance_cvxpy(sigma, bounds=None, long_only=True):
    """
    Global minimum-variance portfolio via CVXPY.
        min  w^T Sigma w
        s.t. 1^T w = 1, plus optional bounds
    """
    n = sigma.shape[0]
    w = cp.Variable(n)

    # omega^T*Sigma*omega, CVXPY recognizes it as a convex quadratic and dispatches to a real QP solver.
    # CVXPY *knows* it's a convex QP, so it uses OSQP's interior-point method rather than treating it as a generic nonlinear problem the way SLSQP does.
    # CVXPY checks convexity by checking Σ⪰0, and a slightly-negative eigenvalue makes it *refuse to solve*, throwing "Problem does not follow DCP rules." 
    # PSD-wrap Sigma so CVXPY recognizes the QP as convex even if the input has tiny negative eigenvalues from estimation noise. `psd_wrap` tells CVXPY "trust me, treat this as PSD.
    objective = cp.Minimize(cp.quad_form(w, cp.psd_wrap(sigma)))

    constraints = [cp.sum(w) == 1]
    if long_only:
        constraints.append(w >= 0)
    if bounds is not None:
        lo, hi = bounds
        constraints += [w >= lo, w <= hi]

    prob = cp.Problem(objective, constraints)
    # For pure QPs it's excellent. If you later add second-order-cone constraints (e.g., an explicit risk cap rather than risk in the objective), switch to `cp.ECOS`, also free.
    prob.solve(solver=cp.OSQP)            # free QP solver, no license

    if prob.status not in ("optimal", "optimal_inaccurate"):
        raise ValueError(f"solve failed: {prob.status}")
    return w.value
min_variance_cvxpy(sigma)

array([0.26315789, 0.47953216, 0.25730994])

Two-fund theorem: $w=g+\mu_p h$ is affine in the target return $\mu_p$. Two fixed vectors $g$ and $h$ generate the entire frontier as you vary $\mu_p$.
Grouping by two frontier portfolios $g$ and $g+h$, $w=(10\mu_p)g+\mu_p(g+h)$, and every other froniter portfolio is a combination of them.
Plugging $w=g+\mu_p h$ into $\sigma_p^2=w^T \Sigma w$ gives the quadratic $\sigma_p^2={{A \mu_p^2 - 2B\mu_p+c} \over D}$